<a href="https://colab.research.google.com/github/natchanant-arch/Project_Savings_Cooperative/blob/%E0%B8%AD%E0%B8%B1%E0%B8%99%E0%B8%AD%E0%B8%B1%E0%B8%99/Cooperative_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏦 ธุรกิจสหกรณ์ออมทรัพย์ (Savings Cooperative)

ระบบจำลองบัญชีสหกรณ์ออมทรัพย์ ครอบคลุมการ **ฝาก – ถอน – โอน** โดยสมาชิกทำธุรกรรมได้ทีละรายการ พร้อมตรวจสอบยอดเงินคงเหลือให้เพียงพอก่อนทำรายการทุกครั้ง และคำนวณดอกเบี้ยจากยอดเงินคงเหลือในบัญชีเมื่อสิ้นปี

---

## 📑 สารบัญ

| ส่วน | หัวข้อ |
|:---:|---|
| 0 | Import เพื่อเรียกใช้งานชุดคำสั่ง / ฟังก์ชันสำเร็จรูป |
| 1 | เตรียม Class และฟังก์ชัน |
| 2 | ทดสอบฟังก์ชันทีละตัว ก่อนประกอบเป็นกระบวนการ |
| 3 | ฟังก์ชันอธิบายขั้นตอนคำนวณรายการ |
| 4 | จำลอง "ลูกค้า 1 คนเดินเข้าธนาคาร" แบบ step-by-step |
| 5 | จำลองลูกค้าหลายคนเดินเข้าธนาคารต่อเนื่องกัน |
| 6 | สรุปผล — ยืนยันว่าฟังก์ชันคืนค่าถูกต้องและใช้ต่อได้จริง |
| 7 | ตารางลูกค้า |
| 8 | ตารางธุรกรรม |
| 9 | สรุปผลรวม 300 รายการ |

---

## — Import เพื่อ เรียกใช้งานชุดคำสั่ง หรือฟังก์ชันสำเร็จรูป —

In [ ]:
import random
import time
from datetime import datetime, timedelta
!pip install Faker
from faker import Faker
fake = Faker("th_TH")
random.seed(1)
import pandas as pd
import matplotlib, os, shutil
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns

In [ ]:
# 1. ติดตั้งฟอนต์ภาษาไทย
!apt-get -y install fonts-thai-tlwg

# 2. ล้าง cache ของ matplotlib เพื่ออัปเดตฟอนต์ใหม่
cache_dir = matplotlib.get_cachedir()
if os.path.exists(cache_dir):
    shutil.rmtree(cache_dir)

# 3. ลงทะเบียนฟอนต์ใหม่เข้ากับ FontManager
font_path = '/usr/share/fonts/truetype/tlwg/Loma.ttf'
if os.path.exists(font_path):
    fm.fontManager.addfont(font_path)

# 4. ตั้งค่าฟอนต์หลัก
plt.rcParams['font.family'] = 'Loma'
plt.rcParams['axes.unicode_minus'] = False

print("ตั้งค่าระบบฟอนต์ภาษาไทยสำเร็จ!")

---

## ส่วนที่ 1 — เตรียม Class และฟังก์ชัน

มีทั้งหมด 3 คลาส คือ

1. **Class Member** (สมาชิก) — รหัสสมาชิก, ชื่อสมาชิก, เลขบัตรประจำตัวประชาชน, เบอร์โทรศัพท์
2. **Class Account** (บัญชีออมทรัพย์) — เลขบัญชี, ยอดเงินคงเหลือ, เจ้าของบัญชี, ดอกเบี้ยต่อปี
3. **Class Transaction** (ธุรกรรม) — หมายเลขธุรกรรม, บัญชี, ประเภทธุรกรรม, จำนวนเงิน, บัญชีปลายทาง

In [ ]:
class Member:
    """ข้อมูลสมาชิกธนาคารออมทรัพย์"""

    def __init__(
        self,
        member_id,
        customer_name,
        citizen_id=None,
        phone_number=None,
    ):
        self.member_id = member_id
        self.customer_name = customer_name
        self.citizen_id = citizen_id
        self.phone_number = phone_number

    # แสดงข้อมูลสมาชิก
    def get_info(self):
        return (
            f"ลูกค้า ID: {self.member_id} | ชื่อ: {self.customer_name} | "
            f"เลขบัตรประชาชน: {self.citizen_id} | เบอร์โทร: {self.phone_number}"
        )

    # อัปเดตข้อมูลส่วนตัว
    def update_phone(self, new_phone):
        self.phone_number = new_phone

In [ ]:
class Account:
    """บัญชีออมทรัพย์"""

    def __init__(
        self,
        account_number,
        balance,
        owner,
        interest_rate=0.015,
    ):
        self.account_number = account_number
        self.balance = float(balance)
        self.owner = owner
        self.interest_rate = interest_rate

    def deposit(self, amount):
        """ฝากเงิน: balance = balance + amount"""
        self.balance += amount
        return "ฝากเงินสำเร็จ"

    def withdraw(self, amount):
        """ถอนเงิน: ตรวจสอบ balance >= amount"""
        if self.balance < amount:
            return f"ยอดเงินไม่พอ (มีอยู่ {self.balance:,.2f} บาท)"

        self.balance -= amount
        return "ถอนเงินสำเร็จ"

    def transfer(self, target_account, amount):
        """โอนเงิน: ตัดบัญชีต้นทาง และบวกเข้าบัญชีปลายทาง"""
        if self.balance < amount:
            return f"ยอดเงินไม่พอโอน (มีอยู่ {self.balance:,.2f} บาท)"

        self.balance -= amount
        target_account.balance += amount
        return "โอนเงินสำเร็จ"

    def apply_interest(self):
        """คำนวณดอกเบี้ยและบวกเข้ายอดคงเหลือ"""
        interest = self.balance * self.interest_rate
        self.balance += interest
        return interest

In [ ]:
def generate_transaction_data(txn_id):
    """ฟังก์ชันสุ่มคิว 40 รายการต่อวัน (รับแค่ txn_id)"""
    base_date = datetime(2026, 8, 22)
    day_idx = 0
    total_items = 0

    while True:
        items_today = 40

        # เช็คว่า txn_id นี้ยังอยู่ในโควตาสะสมของวันนี้หรือไม่
        if txn_id <= total_items + items_today:
            queue_num = txn_id - total_items
            current_date = base_date + timedelta(days=day_idx)

            return {
                "วันที่": current_date.strftime("%d/%m/%Y"),
                "หมายเลขคิว": f"A-{queue_num:03d}",
                "queue_seq": queue_num - 1,
            }

        total_items += items_today
        day_idx += 1

In [ ]:
class Transaction:

    def __init__(
        self,
        txn_id,
        account,
        transaction_type,
        amount,
        target_account=None,
    ):
        self.txn_id = txn_id

        date_info = generate_transaction_data(txn_id)
        self.queue_number = date_info["หมายเลขคิว"]
        self.txn_date = date_info["วันที่"]

        base_start_time = datetime.strptime("08:30:00", "%H:%M:%S")
        queue_seq = date_info["queue_seq"]

        minutes_added = (queue_seq * random.randint(8, 11)) + random.randint(
            0, 2
        )
        seconds_added = random.randint(0, 59)

        actual_time = base_start_time + timedelta(
            minutes=minutes_added, seconds=seconds_added
        )
        self.time = actual_time.strftime("%H:%M:%S")

        self.account = account
        self.account_number = account.account_number
        self.customer_name = account.owner.customer_name
        self.transaction_type = transaction_type
        self.amount = amount
        self.target_account = target_account

    def to_dict(self):
        interest = self.account.balance * self.account.interest_rate

        if isinstance(self.customer_name, (tuple, list)):
            fname, lname = self.customer_name[0], self.customer_name[1]
        else:
            parts = str(self.customer_name).split(" ", 1)
            fname = parts[0]
            lname = parts[1] if len(parts) > 1 else "-"

        if self.target_account:
            target_acc_no = getattr(
                self.target_account, "account_number", str(self.target_account)
            )
        else:
            target_acc_no = "-"

        return {
            "ID รายการ": self.txn_id,
            "หมายเลขคิว": self.queue_number,
            "วันที่ทำรายการ": self.txn_date,
            "เวลาทำรายการ": self.time,
            "เลขบัญชี": self.account_number,
            "ชื่อ": fname,
            "นามสกุล": lname,
            "ประเภทรายการ": self.transaction_type,
            "จำนวนเงิน": self.amount,
            "บัญชีปลายทาง": target_acc_no,
            "ยอดหลังทำรายการ": round(self.account.balance, 2),
            "ดอกเบี้ยสิ้นปี (1.5%)": round(interest, 2),
            "ยอดรวมดอกเบี้ยสุทธิ": round(self.account.balance + interest, 2),
        }

# ส่วนที่ 2 — ฟังก์ชันช่วยงาน (Helper Function)

In [ ]:
def generate_thai_name():
    """ฟังก์ชัน: สุ่มชื่อและนามสกุลลูกค้าแยกกัน"""
    name = fake.name()
    first_name, last_name = name.split(" ", 1)
    return f"{first_name} {last_name}"


def random_amount(min_val=100.0, max_val=2000.0):
    """ฟังก์ชัน: สุ่มยอดเงิน -> คืนค่าเป็น float"""
    return round(random.uniform(min_val, max_val), 2)


def format_currency(amount, symbol="บาท"):
    """ฟังก์ชัน: จัดรูปแบบตัวเลขเป็นสตรีงราคา -> คืนค่าเป็น string"""
    return f"{amount:,.2f} {symbol}"

## ส่วนที่ 3 — ฟังก์ชันอธิบายขั้นตอนคำนวณราคา

In [ ]:
def explain_transaction_calculation(transaction):
    """ฟังก์ชันคำนวณเงิน ฝาก/ถอน/โอน และเรียกใช้ Method ของ Account"""

    account = transaction.account
    amount = float(transaction.amount)
    txn_type = transaction.transaction_type

    print(f"หมายเลขคิว = '{transaction.queue_number}'")
    print(f"หมายเลขบัญชี = '{account.account_number}'")
    print(f"ชื่อลูกค้า = '{transaction.customer_name}'")
    print(f"ประเภทรายการ = '{txn_type}'")
    print(f"ยอดเงินก่อนทำรายการ = {format_currency(account.balance)}")

    if txn_type == "ฝากเงิน":
        status_msg = account.deposit(amount)

    elif txn_type == "ถอนเงิน":
        status_msg = account.withdraw(amount)
        if "ยอดเงินไม่พอ" in status_msg:
            print(f"จำนวนเงินทำรายการ = {format_currency(amount)}")
            print(f"ยอดเงินคงเหลือหลังทำรายการ = {format_currency(account.balance)}")
            print(f"สถานะรายการ = '{status_msg}'")
            print("❌ ทำรายการไม่สำเร็จ!")
            return False

    elif txn_type == "โอนเงิน":
        if transaction.target_account:
            print(f"บัญชีปลายทาง = '{transaction.target_account}'")

        dummy_member = Member(0, "บัญชีปลายทาง")
        dummy_target = Account("987-6-00000-0", balance=0.0, owner=dummy_member)
        status_msg = account.transfer(dummy_target, amount)
        if "ยอดเงินไม่พอ" in status_msg:
            print(f"จำนวนเงินทำรายการ = {format_currency(amount)}")
            print(f"ยอดเงินคงเหลือหลังทำรายการ = {format_currency(account.balance)}")
            print(f"สถานะรายการ = '{status_msg}'")
            print("❌ ทำรายการไม่สำเร็จ!")
            return False

    interest_val = account.apply_interest()

    print(f"จำนวนเงินทำรายการ = {format_currency(amount)}")
    print(f"ยอดเงินคงเหลือหลังทำรายการ = {format_currency(account.balance)}")
    print(f"สถานะรายการ = '{status_msg}'")
    print(f"ดอกเบี้ยที่ได้รับเมื่อสิ้นปี (1.5%) = {format_currency(interest_val)}")

    return True

In [ ]:
import random

random.seed(1)
fake.seed_instance(1)

transactions = []

for i in range(1, 301):
    name = generate_thai_name()
    amount = random_amount()
    service = random.choice(["ฝากเงิน", "ถอนเงิน", "โอนเงิน"])

    member = Member(member_id=1 + i, customer_name=name)

    initial_balance = round(random.uniform(100, 3000), 2)

    acc_p1 = random.randint(100, 999)
    acc_p2 = random.randint(1, 9)
    acc_p3 = random.randint(10000, 99999)
    random_account_no = f"{acc_p1}-{acc_p2}-{acc_p3:05d}-0"

    account = Account(account_number=random_account_no, balance=initial_balance, owner=member)
    target_acc = f"987-6-{random.randint(10000, 99999)}-0" if service == "โอนเงิน" else None

    if service == "ฝากเงิน":
        account.deposit(amount)
    elif service == "ถอนเงิน":
        account.withdraw(amount)
    elif service == "โอนเงิน":
        dummy_mem = Member(0, "ปลายทาง")
        dummy_acc = Account("987-6-00000-0", balance=0.0, owner=dummy_mem)
        account.transfer(dummy_acc, amount)

    daily_queue = ((i - 1) % 40) + 1
    queue_no = f"A-{daily_queue:03d}"

    account.apply_interest()

    transaction = Transaction(
        txn_id=i,
        account=account,
        transaction_type=service,
        amount=amount,
        target_account=target_acc
    )

    transactions.append(transaction)

print("\n--- [ตัวอย่างการแสดงผลรายการแรก (คิว A-001)] ---")
explain_transaction_calculation(transactions[0])

> 💡 **หมายเหตุ:** เป็นการล็อกค่าของการสุ่ม (Random Seed) ไว้ เพื่อให้ทุกครั้งที่กดรันโปรแกรม ระบบจะสุ่มได้ตัวเลขและข้อมูลชุดเดิมเสมอ ทำให้ง่ายต่อการทดสอบและตรวจสอบความถูกต้องของระบบ